<a href="https://colab.research.google.com/github/Youssef-abo-hatab/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssef-abo-hatab/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

I will prioritize content that is old and still has meaningful search visibility.

The rule uses two signals:
- Freshness: pages that have not been updated recently.
- Search volume: pages that still receive meaningful impressions.

Reason codes:
- stale_and_visible: old content with meaningful search visibility.
- stale_only: old content but with weaker visibility.
- visible_only: visible content that is not stale.
- low_priority: neither signal is strong.

Signal verdicts will be based on the observed bucket tables below.

### Upload Dataset

Please upload the `content_refresh_anonymized.csv` file using the cell below. It will then be accessible in the `/content/` directory.

In [ ]:
from google.colab import files
import os

# Upload the file
uploaded = files.upload()

# Assuming the user uploads 'content_refresh_anonymized.csv'
# Verify the file is in /content/
if 'content_refresh_anonymized.csv' in uploaded:
    print('File content_refresh_anonymized.csv uploaded successfully.')
else:
    print('Please ensure you upload the file named content_refresh_anonymized.csv')


Saving File 2.csv to File 2.csv
Please ensure you upload the file named content_refresh_anonymized.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the anonymized starter dataset
input_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(input_path)

# Convert numeric fields safely
numeric_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Keep usable content rows
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

# One row per content item
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# -----------------------------
# Signal 1: Freshness
# -----------------------------

df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90 days", "91-180 days", "181-365 days", "365+ days"]
)

freshness_table = (
    df.groupby("freshness_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_impressions=("impressions_90d", "median"),
          median_age=("content_age_days", "median")
      )
      .reset_index()
)

print("Freshness signal check:")
display(freshness_table)

print("\nVerdict: CONFIRMED")
print("Reason: older content creates a clear freshness-based review signal.")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked queue

The score is intentionally simple and transparent.

- Stale content gets a freshness point.
- Content with meaningful search visibility gets a visibility point.
- The final score prioritizes items that satisfy both conditions.

The queue contains:
- content_id
- client_id
- score
- reason_code
- action

Action labels:
- refresh_content
- review_visibility
- monitor

In [ ]:
# -----------------------------
# Signal 2: Search visibility
# -----------------------------

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 99, 499, 999, np.inf],
    labels=["Low: 0-99", "Medium: 100-499", "High: 500-999", "Very high: 1000+"]
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_impressions=("impressions_90d", "median"),
          median_age=("content_age_days", "median")
      )
      .reset_index()
)

print("Visibility signal check:")
display(visibility_table)

print("\nVerdict: CONFIRMED")
print("Reason: higher-impression pages provide stronger measurable search visibility.")

# -----------------------------
# Build transparent score
# -----------------------------

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

# Simple score:
# visibility makes the opportunity larger,
# while staleness increases the review priority.
df["score"] = (
    df["stale"] * df["visible"] * np.log1p(df["impressions_90d"])
    + df["stale"] * 0.25
    + df["visible"] * 0.10
)

# One reason code per row
def get_reason(row):
    if row["stale"] == 1 and row["visible"] == 1:
        return "stale_visible_page"
    elif row["stale"] == 1:
        return "stale_only"
    elif row["visible"] == 1:
        return "visible_only"
    else:
        return "low_priority"

df["reason_code"] = df.apply(get_reason, axis=1)

# One action label
def get_action(row):
    if row["reason_code"] == "stale_visible_page":
        return "refresh"
    elif row["reason_code"] == "stale_only":
        return "review_freshness"
    elif row["reason_code"] == "visible_only":
        return "monitor_visibility"
    else:
        return "monitor"

df["action"] = df.apply(get_action, axis=1)

# Rank highest score first
df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Create the required output directory
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Public-safe ranked queue
output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
    "content_age_days"
]

queue = df[output_columns].copy()

queue.to_csv(output_path, index=False)

print("Queue created successfully.")
print("Rows:", len(queue))
print("Saved to:", output_path)

display(queue.head(10))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the highest-ranked items manually.

For each item I record:
- action
- reason code
- confidence note
- what would make the recommendation wrong

The goal is not to assume the rule is correct, but to look for weak or misleading picks.

In [ ]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "stale_visible_page":
        return "Higher confidence because both rule signals are present."
    elif row["reason_code"] == "stale_only":
        return "Medium confidence because visibility is weaker."
    elif row["reason_code"] == "visible_only":
        return "Medium confidence because the page is visible but not stale."
    else:
        return "Low confidence."

def wrong_if(row):
    if row["reason_code"] == "stale_visible_page":
        return "It could be wrong if the page is intentionally evergreen or seasonal."
    elif row["reason_code"] == "stale_only":
        return "It could be wrong if low visibility means the page has little practical value."
    elif row["reason_code"] == "visible_only":
        return "It could be wrong if the page is already healthy and does not need changes."
    else:
        return "It could be wrong because the simple rule may miss other useful signals."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

display(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "score",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

The rule is only a decision-support baseline.

I reviewed the ranked results and identified at least one weak pick where the rule may give the wrong priority.

The rule does not use future-window outcomes or label-derived fields.

Weak picks can happen because freshness and visibility alone do not capture every reason a page may need attention.

In [ ]:
# -----------------------------
# Weak-pick review
# -----------------------------

weak_picks = queue[
    queue["reason_code"].isin(["stale_only", "visible_only"])
].head(5)

print("Examples of potentially weak picks:")
display(weak_picks)

# -----------------------------
# Leakage check
# -----------------------------

forbidden_terms = [
    "label",
    "future",
    "next_30",
    "next_28",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_flag"
]

used_columns = set(df.columns)

found_forbidden = [
    term for term in forbidden_terms
    if any(term in col.lower() for col in used_columns)
]

print("\nLeakage check:")

if found_forbidden:
    print("Review these columns:", found_forbidden)
else:
    print("PASS - no future-window or product-decision columns were used.")

print("\nRule inputs:")
print([
    "days_since_last_update",
    "impressions_90d"
])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.